In [7]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

### TASK 2 - RECRUSIVE UTILITY FUNCTIONS

In [8]:
# Task 2.1 - Recursive Directory Search

def find_log_files(folder: Path) -> list[Path]:
    """
    Finner alle loggfilene i en mappe og undermappene.

    Args:
        folder (Path): Mappen som skal undersøkes.

    Returns:
        list[Path]: En liste med filbaner til loggfilene.
    """

    folder = Path(folder)
    log_files = []

    # Går gjennom alle elementene i mappen
    for item in folder.iterdir():

        # Hvis elementet er en mappe, undersøkes den rekursivt
        if item.is_dir():
            log_files.extend(find_log_files(item))

        # Hvis elementet er en .log-fil, legges den til i listen
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files


# Task 2.2 - Recursive JSON Traversal

def find_active_registers(data: object) -> set[str]:
    """
    Finner ID-ene til aktive kasseregistre i en datastruktur.
    Funksjonen undersøker dictionaries og lister rekursivt.

    Args:
        data (object): Dataene som skal undersøkes.

    Returns:
        set[str]: Et sett med ID-er til aktive kasseregistre.
    """

    active_registers = set()

    # Hvis data er en dictionary, undersøkes innholdet
    if isinstance(data, dict):

        # Legger til ID-en dersom kasseregisteret er aktivt
        if data.get("active") is True and "id" in data:
            active_registers.add(data["id"])

        # Undersøker alle verdiene i dictionaryen rekursivt
        for value in data.values():
            active_registers.update(find_active_registers(value))

    # Hvis data er en liste, undersøkes hvert element
    elif isinstance(data, list):
        for item in data:
            active_registers.update(find_active_registers(item))

    return active_registers

### TASK 1 - FILE READING AND TRANSACTION PARSING

In [9]:
# Task 1.1

def fileread(filepath: str | Path) -> tuple[str, str]:
    """
    Read a text file using UTF-8 or Latin-1 encoding.

    Args:
        filepath (str | Path): Path to the file.

    Returns:
        tuple[str, str]: File content and the encoding used.
    """

    path = Path(filepath)

    # Try reading the file with UTF-8 encoding first
    try:
        with open(path, "r", encoding="utf-8") as file:
            content = file.read()
            encoding_used = "utf-8"

    # If UTF-8 fails, try Latin-1 encoding instead
    except UnicodeDecodeError:
        with open(path, "r", encoding="latin-1") as file:
            content = file.read()
            encoding_used = "latin-1"

    # Return both the file content and the encoding used
    return content, encoding_used

# Task 1.2

def parse_transactions(content: str) -> list[dict]:
    """
    Extract transactions from log file content using regex.

    Args:
        content (str): Text content from a log file.

    Returns:
        list[dict]: List of dictionaries containing transaction data.
    """

    # Create an empty list to store the transactions
    transactions = []

    # Go through the log file one line at a time
    for line in content.splitlines():

        # Search for transaction data using six capture groups
        match = re.search(
            r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s\[(\w+)\]\sitem=(\w+)\sprice=(\d+\.\d{2})\s([A-Z]{3})\s\(([A-Z]+)\)",
            line
        )

        # Only process lines that match the transaction pattern
        if match:

            # Store the extracted values in a dictionary
            values = {
                "timestamp": match.group(1),
                "register_id": match.group(2),
                "item": match.group(3),
                "price": float(match.group(4)),
                "currency": match.group(5),
                "status": match.group(6)
            }

            # Add the transaction dictionary to the list
            transactions.append(values)

    # Return all extracted transactions
    return transactions

### TASK 3 - NUMPY DATA PROCESSING

In [10]:
# Task 3.1 - Outlier Detection and Correction

def correct_outliers(prices: np.ndarray) -> np.ndarray:
    """
    Finner priser som avviker mer enn tre standardavvik fra gjennomsnittet
    og erstatter dem med medianen av de gyldige prisene.

    Args:
        prices (np.ndarray): En array med priser.

    Returns:
        np.ndarray: En array med korrigerte priser.
    """

    prices = np.array(prices, dtype=float, copy=True)

    # Returnerer en tom array dersom ingen priser er oppgitt
    if prices.size == 0:
        return prices

    # Beregner gjennomsnittet og standardavviket
    mean = np.mean(prices)
    std = np.std(prices)

    # Finner priser som ligger mer enn tre standardavvik fra gjennomsnittet
    outliers = np.abs(prices - mean) > (3 * std)

    # Finner prisene som ikke er outliers
    valid_prices = prices[~outliers]

    # Beregner medianen av de gyldige prisene
    median = np.median(valid_prices)

    # Erstatter outliers med medianen
    prices[outliers] = median

    return prices


# Task 3.2 - Refund Ratio

def compute_refund_ratio(
    sales_count: np.ndarray,
    refund_count: np.ndarray
) -> np.ndarray:
    """
    Calculate the element-wise refund ratio using NumPy arrays.
    Adds a small epsilon to prevent division by zero.
    Returns an array containing the calculated refund ratios.
    """

    # Small value to prevent division by zero
    epsilon = 1e-6

    # Calculate refund ratio for each element
    ratio = refund_count / (sales_count + epsilon)

    return ratio

### TASK 4 - PANDAS DATA ANALYSIS

In [11]:
# Task 4.1 - Create DataFrame

def create_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """
    Gjør en liste med transaksjoner om til en DataFrame.

    Args:
        transactions (list[dict]): Liste med transaksjonsdata.

    Returns:
        pd.DataFrame: Transaksjoner sortert etter tidspunkt.
    """

    # Lager en DataFrame fra transaksjonslisten
    df = pd.DataFrame(transactions)

    # Gjør tidspunktet om til datetime-format
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # Bruker tidspunktet som indeks
    df = df.set_index("timestamp")

    # Sorterer transaksjonene kronologisk
    df = df.sort_index()

    return df


# Task 4.2 - Hourly Revenue

def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """
    Beregner summen av transaksjonspriser per time og produkt.

    Args:
        df (pd.DataFrame): DataFrame med transaksjoner.

    Returns:
        pd.DataFrame: Summert pris per time og produkt.
    """

    # Grupperer etter produkt og summerer prisene per time
    hourly = df.groupby("item").resample("1h")["price"].sum()

    # Gjør resultatet om til en DataFrame
    return hourly.to_frame(name="price")


# Task 4.3 - Transaction Pivot Table

def transaction_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a pivot table showing total transaction revenue per item and status.
    Returns a DataFrame with items as rows and transaction statuses as columns.
    """

    # Group transactions by item and status, summing the prices
    pivot = pd.pivot_table(
        df,
        index="item",
        columns="status",
        values="price",
        aggfunc="sum",
        fill_value=0
    )

    return pivot


# Task 4.4 - Refund Ratio per Item

def refund_ratio_per_item(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate the refund ratio per item using the function from Task 3.2.
    Returns a DataFrame containing the refund ratio for each item.
    """

    # Count transactions for each item and status
    counts = pd.crosstab(df["item"], df["status"])

    # Get the number of successful sales and refunds per item
    sales_count = counts.get("SUCCESS", pd.Series(0, index=counts.index))
    refund_count = counts.get("REFUND", pd.Series(0, index=counts.index))

    # Calculate refund ratios using the function from Task 3.2
    ratios = compute_refund_ratio(
        sales_count.to_numpy(),
        refund_count.to_numpy()
    )

    # Return a DataFrame with item names and refund ratios
    return pd.DataFrame({"refund_ratio": ratios}, index=counts.index)